# Project 1: Loan Default Risk & Portfolio Analytics

**Question:** Which borrowers default, why, and what approval policy makes the lender more profit, not just fewer defaults?

**Stack:** SQL (SQLite) · Python (pandas, scikit-learn, matplotlib) · Excel · Power BI

**Note:** the data is synthetic (modelled on Lending Club fields) so anyone can reproduce it. Re-run on real data and quote your own numbers.

## 0. Setup
Run this notebook **top to bottom** (Run All). It creates the folders `data/`, `sql/` and `outputs/` next to the notebook.

First install the libraries (only needed once):

In [ ]:
# !pip install pandas numpy scikit-learn matplotlib openpyxl

In [ ]:
import sys, json, re, sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
for d in ['data', 'sql', 'outputs/charts', 'outputs/powerbi', 'outputs/sql_results']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('Working folder:', Path.cwd())

## 1. Create the dataset
Synthetic data with realistic patterns, saved to `data/loans.csv`. To use real Kaggle data instead, skip this cell and put your file in `data/` (see the project README for the column mapping).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
N = 50_000
OUT = Path.cwd() / "data" / "loans.csv"

rng = np.random.default_rng(SEED)

# --- 1. A hidden "true credit risk" for every borrower (higher = riskier) -----------
risk = rng.normal(0, 1, N)

# Lender-assigned grade A-G is a noisy view of that hidden risk
grade_idx = np.digitize(risk + rng.normal(0, 0.35, N), [-1.2, -0.5, 0.1, 0.7, 1.2, 1.7])
grades = np.array(list("ABCDEFG"))[grade_idx]

# Interest rate rises with grade
int_rate = np.clip(6.5 + 2.6 * grade_idx + rng.normal(0, 0.8, N), 5.3, 28.0).round(2)

# --- 2. Borrower attributes ----------------------------------------------------------
income = np.exp(rng.normal(11.05, 0.5, N) - 0.08 * risk)
income = income.round(-2)
income[rng.choice(N, 40, replace=False)] *= 15  # a few extreme outliers to clean later

dti = np.clip(rng.normal(17, 7, N) + 1.5 * risk, 0, 45).round(2)
loan_amount = (np.clip(np.exp(rng.normal(9.6, 0.55, N)), 1_000, 40_000) // 25 * 25).astype(int)
term = np.where(rng.random(N) < np.clip(loan_amount / 60_000, 0.1, 0.6), 60, 36)
emp_length = rng.choice(np.arange(0, 11), N, p=[.08, .08, .09, .09, .07, .07, .05, .05, .05, .04, .33])
home = rng.choice(["RENT", "MORTGAGE", "OWN"], N, p=[.42, .45, .13])
purpose = rng.choice(
    ["debt_consolidation", "credit_card", "home_improvement", "small_business",
     "major_purchase", "medical", "other"],
    N, p=[.55, .20, .07, .04, .05, .03, .06])
state_w = np.array([.17, .10, .11, .09, .05, .05, .05, .05, .04, .04, .03, .03])
state = rng.choice(["CA", "TX", "NY", "FL", "IL", "PA", "OH", "GA", "NC", "NJ", "VA", "WA"], N,
                   p=state_w / state_w.sum())
revol_util = np.clip(rng.normal(50, 22, N) + 6 * risk, 0, 100).round(1)
delinq = rng.poisson(0.2 + 0.25 * np.clip(risk, 0, None))

start = pd.Timestamp("2015-01-01")
issue_date = start + pd.to_timedelta(rng.integers(0, 4 * 365, N), unit="D")  # DatetimeIndex
issue_year = issue_date.year.to_numpy()

# --- 3. Default outcome: logistic relationship with the drivers ----------------------
purpose_effect = pd.Series(purpose).map({
    "small_business": 0.55, "medical": 0.25, "debt_consolidation": 0.05, "credit_card": -0.10,
    "home_improvement": -0.20, "major_purchase": -0.15, "other": 0.10}).to_numpy()
logit = (
    -2.65
    + 0.28 * grade_idx
    + 0.040 * (dti - 17)
    + 0.30 * (term == 60)
    - 0.45 * (np.log(income) - 11.05)
    + 0.010 * (revol_util - 50)
    + 0.22 * delinq
    - 0.035 * emp_length
    + 0.14 * (home == "RENT")
    + purpose_effect
    + 0.00001 * (loan_amount - 14_800)
    + 0.015 * (issue_year - 2016)
)
p_default = 1 / (1 + np.exp(-logit))
defaulted = rng.random(N) < p_default

# --- 4. Cash flows (lets us calculate profit per loan) -------------------------------
interest_earned = loan_amount * (int_rate / 100) * (term / 12) * 0.55   # ~avg balance factor
paid_before_default = rng.uniform(0.15, 0.55, N)   # share of principal recovered on defaulted loans
total_payment = np.where(defaulted, loan_amount * paid_before_default,
                         loan_amount + interest_earned).round(2)

df = pd.DataFrame({
    "loan_id": np.arange(100_000, 100_000 + N),
    "issue_date": issue_date.strftime("%Y-%m-%d"),
    "loan_amount": loan_amount,
    "term_months": term,
    "int_rate": int_rate,
    "grade": grades,
    "annual_income": income,
    "dti": dti,
    "emp_length_yrs": emp_length.astype(float),
    "home_ownership": home,
    "purpose": purpose,
    "state": state,
    "revol_util": revol_util,
    "delinq_2yrs": delinq,
    "loan_status": np.where(defaulted, "Charged Off", "Fully Paid"),
    "total_payment": total_payment,
})

# --- 5. Make the data realistically messy (so cleaning is meaningful) ----------------
df.loc[rng.choice(N, int(N * 0.05), replace=False), "emp_length_yrs"] = np.nan
df.loc[rng.choice(N, int(N * 0.01), replace=False), "revol_util"] = np.nan
dupes = df.sample(int(N * 0.003), random_state=SEED)
df = pd.concat([df, dupes], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

OUT.parent.mkdir(exist_ok=True)
df.to_csv(OUT, index=False)
print(f"Saved {len(df):,} rows -> {OUT}")
print(f"Overall default rate: {defaulted.mean():.1%}")

## 2. SQL analysis
The 11 queries live in one text block. They are saved to `sql/queries.sql`, loaded into an in-memory-style SQLite database, and run one by one. The same SQL works in PostgreSQL/MySQL with tiny changes.

In [ ]:
SQL_TEXT = r"""
-- =====================================================================================
-- PROJECT 1: LOAN DEFAULT RISK  |  SQL ANALYSIS
-- Dialect: SQLite (runs anywhere). Syntax is standard - works in PostgreSQL/MySQL 8+
-- with tiny changes (noted inline).
--
-- `loans_v` is a VIEW over the raw `loans` table that adds:
--     is_default = 1 if loan_status = 'Charged Off' else 0
--     issue_year / issue_quarter extracted from issue_date
--
-- Each query starts with "-- name:" so run_sql.py can execute and export them one by one.
-- =====================================================================================


-- name: 01_portfolio_kpis
-- question: What does the overall portfolio look like?
SELECT
    COUNT(*)                                            AS total_loans,
    ROUND(SUM(loan_amount) / 1e6, 2)                    AS total_disbursed_mn,
    ROUND(AVG(int_rate), 2)                             AS avg_interest_rate_pct,
    ROUND(100.0 * AVG(is_default), 2)                   AS default_rate_pct,
    ROUND(SUM(CASE WHEN is_default = 1 THEN loan_amount END) / 1e6, 2) AS defaulted_amount_mn,
    ROUND(SUM(total_payment - loan_amount) / 1e6, 2)    AS net_profit_mn
FROM loans_v;


-- name: 02_default_rate_by_grade
-- question: Does the lender's grade actually rank risk correctly?
SELECT
    grade,
    COUNT(*)                                  AS loans,
    ROUND(AVG(int_rate), 2)                   AS avg_int_rate,
    ROUND(100.0 * AVG(is_default), 2)         AS default_rate_pct,
    ROUND(AVG(total_payment - loan_amount), 0) AS avg_profit_per_loan
FROM loans_v
GROUP BY grade
ORDER BY grade;


-- name: 03_default_rate_by_dti_band
-- question: How does debt-to-income (DTI) relate to default? (CASE WHEN banding)
SELECT
    CASE
        WHEN dti < 10 THEN '1) <10'
        WHEN dti < 20 THEN '2) 10-20'
        WHEN dti < 30 THEN '3) 20-30'
        ELSE               '4) 30+'
    END                                       AS dti_band,
    COUNT(*)                                  AS loans,
    ROUND(100.0 * AVG(is_default), 2)         AS default_rate_pct
FROM loans_v
GROUP BY dti_band
ORDER BY dti_band;


-- name: 04_default_rate_by_purpose
-- question: Which loan purposes are riskiest? (HAVING filters out tiny groups)
SELECT
    purpose,
    COUNT(*)                                  AS loans,
    ROUND(100.0 * AVG(is_default), 2)         AS default_rate_pct,
    ROUND(SUM(CASE WHEN is_default = 1 THEN loan_amount END) / 1e6, 2) AS defaulted_amount_mn
FROM loans_v
GROUP BY purpose
HAVING COUNT(*) >= 500
ORDER BY default_rate_pct DESC;


-- name: 05_state_risk_ranking
-- question: Rank states by default rate (window function RANK)
WITH state_stats AS (
    SELECT
        state,
        COUNT(*)                          AS loans,
        AVG(is_default)                   AS default_rate
    FROM loans_v
    GROUP BY state
)
SELECT
    state,
    loans,
    ROUND(100 * default_rate, 2)                              AS default_rate_pct,
    RANK() OVER (ORDER BY default_rate DESC)                  AS risk_rank,
    ROUND(100 * (default_rate - (SELECT AVG(is_default) FROM loans_v)), 2) AS gap_vs_portfolio_pp
FROM state_stats
ORDER BY risk_rank;


-- name: 06_vintage_analysis_yearly
-- question: Is underwriting quality getting better or worse? (LAG for year-over-year change)
WITH yearly AS (
    SELECT
        issue_year,
        COUNT(*)                          AS loans,
        ROUND(SUM(loan_amount) / 1e6, 2)  AS disbursed_mn,
        100.0 * AVG(is_default)           AS default_rate_pct
    FROM loans_v
    GROUP BY issue_year
)
SELECT
    issue_year,
    loans,
    disbursed_mn,
    ROUND(default_rate_pct, 2)                                         AS default_rate_pct,
    ROUND(default_rate_pct - LAG(default_rate_pct) OVER (ORDER BY issue_year), 2) AS yoy_change_pp
FROM yearly
ORDER BY issue_year;


-- name: 07_cumulative_disbursement
-- question: Running total of money lent per quarter (SUM() OVER running total)
WITH quarterly AS (
    SELECT
        issue_year,
        issue_quarter,
        SUM(loan_amount) AS disbursed
    FROM loans_v
    GROUP BY issue_year, issue_quarter
)
SELECT
    issue_year || '-Q' || issue_quarter                                        AS quarter,
    ROUND(disbursed / 1e6, 2)                                                  AS disbursed_mn,
    ROUND(SUM(disbursed) OVER (ORDER BY issue_year, issue_quarter) / 1e6, 2)   AS cumulative_mn
FROM quarterly
ORDER BY issue_year, issue_quarter;


-- name: 08_grade_by_term_matrix
-- question: Do 60-month loans hurt more in the weaker grades? (two-way segmentation)
SELECT
    grade,
    ROUND(100.0 * AVG(CASE WHEN term_months = 36 THEN is_default END), 2) AS default_pct_36m,
    ROUND(100.0 * AVG(CASE WHEN term_months = 60 THEN is_default END), 2) AS default_pct_60m,
    COUNT(CASE WHEN term_months = 60 THEN 1 END)                          AS loans_60m
FROM loans_v
GROUP BY grade
ORDER BY grade;


-- name: 09_interest_rate_deciles
-- question: Is the lender being paid enough for risk? (NTILE deciles of interest rate)
WITH ranked AS (
    SELECT
        loan_id, is_default, int_rate, total_payment, loan_amount,
        NTILE(10) OVER (ORDER BY int_rate) AS rate_decile
    FROM loans_v
)
SELECT
    rate_decile,
    ROUND(MIN(int_rate), 1)                      AS min_rate,
    ROUND(MAX(int_rate), 1)                      AS max_rate,
    ROUND(100.0 * AVG(is_default), 2)            AS default_rate_pct,
    ROUND(AVG(total_payment - loan_amount), 0)   AS avg_profit_per_loan
FROM ranked
GROUP BY rate_decile
ORDER BY rate_decile;


-- name: 10_high_risk_segment_rule
-- question: How much damage does a simple rule 'DTI >= 30 AND employed < 2 yrs' cause?
-- (This is the rule tested in the Python policy simulation.)
WITH flagged AS (
    SELECT
        *,
        CASE WHEN dti >= 30 AND emp_length_yrs < 2 THEN 'High-risk rule' ELSE 'Rest of book' END AS segment
    FROM loans_v
)
SELECT
    segment,
    COUNT(*)                                       AS loans,
    ROUND(100.0 * AVG(is_default), 2)              AS default_rate_pct,
    ROUND(AVG(total_payment - loan_amount), 0)     AS avg_profit_per_loan,
    ROUND(SUM(total_payment - loan_amount) / 1e6, 3) AS total_profit_mn
FROM flagged
GROUP BY segment;


-- name: 11_top_risk_combinations
-- question: Which grade x purpose combinations have the worst default rates? (min volume filter)
SELECT
    grade,
    purpose,
    COUNT(*)                          AS loans,
    ROUND(100.0 * AVG(is_default), 2) AS default_rate_pct
FROM loans_v
GROUP BY grade, purpose
HAVING COUNT(*) >= 150
ORDER BY default_rate_pct DESC
LIMIT 10;
"""
(Path("sql") / "queries.sql").write_text(SQL_TEXT)

In [ ]:
import re
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
DB_PATH = ROOT / "data" / "loans.db"
OUT_DIR = ROOT / "outputs" / "sql_results"

VIEW_SQL = """
DROP VIEW IF EXISTS loans_v;
CREATE VIEW loans_v AS
SELECT
    *,
    CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END           AS is_default,
    CAST(strftime('%Y', issue_date) AS INTEGER)                        AS issue_year,
    (CAST(strftime('%m', issue_date) AS INTEGER) + 2) / 3              AS issue_quarter
FROM loans;
"""


def build_database() -> sqlite3.Connection:
    df = pd.read_csv(ROOT / "data" / "loans.csv").drop_duplicates(subset="loan_id")
    conn = sqlite3.connect(DB_PATH)
    df.to_sql("loans", conn, if_exists="replace", index=False)
    conn.executescript(VIEW_SQL)
    print(f"Loaded {len(df):,} loans into {DB_PATH.name}\n")
    return conn


def parse_queries(sql_text: str) -> dict:
    """Split the .sql file into {name: (question, sql)} using the '-- name:' markers."""
    blocks = re.split(r"^-- name:\s*", sql_text, flags=re.MULTILINE)[1:]
    queries = {}
    for block in blocks:
        name, _, rest = block.partition("\n")
        question = re.search(r"^-- question:\s*(.*)$", rest, flags=re.MULTILINE)
        sql = "\n".join(l for l in rest.splitlines() if not l.startswith("--")).strip()
        queries[name.strip()] = (question.group(1) if question else "", sql)
    return queries


def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    conn = build_database()
    queries = parse_queries(SQL_TEXT)
    for name, (question, sql) in queries.items():
        result = pd.read_sql_query(sql, conn)
        result.to_csv(OUT_DIR / f"{name}.csv", index=False)
        print(f"=== {name} ===\n{question}")
        print(result.to_string(index=False), "\n")
    conn.close()

main()

## 3. Cleaning, features, model and simulation code
This cell defines the reusable functions (cleaning, feature engineering, model, simulation). Nothing runs yet.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "loans.csv"
GRADES = list("ABCDEFG")

NUMERIC_FEATURES = ["grade_idx", "dti", "annual_income", "revol_util", "delinq_2yrs",
                    "emp_length_yrs", "loan_amount", "term_months", "loan_to_income"]
CATEGORICAL_FEATURES = ["home_ownership", "purpose"]
RISK_BAND_LABELS = ["Very Low", "Low", "Medium", "High", "Very High"]

# Business assumptions used to estimate EXPECTED profit of a loan (documented in README)
INTEREST_FACTOR = 0.55        # avg share of the loan outstanding over its life (amortising loan)
LOSS_GIVEN_DEFAULT = 0.625    # share of principal lost when a loan defaults (matches generator: 1 - mean recovery 0.375)


# ----------------------------------------------------------------------------------
# 1. CLEANING
# ----------------------------------------------------------------------------------
def load_and_clean(path: Path = DATA_PATH):
    """Return (clean_df, report). Every cleaning decision is logged in `report`."""
    df = pd.read_csv(path, parse_dates=["issue_date"])
    report = {"rows_raw": len(df)}

    # a) duplicates - same loan_id appearing twice
    df = df.drop_duplicates(subset="loan_id").copy()
    report["duplicates_removed"] = report["rows_raw"] - len(df)

    # b) missing values - median imputation (robust to outliers) + a flag so the
    #    information "this was missing" is not lost
    report["emp_length_missing"] = int(df["emp_length_yrs"].isna().sum())
    report["revol_util_missing"] = int(df["revol_util"].isna().sum())
    df["emp_length_missing"] = df["emp_length_yrs"].isna().astype(int)
    df["emp_length_yrs"] = df["emp_length_yrs"].fillna(df["emp_length_yrs"].median())
    df["revol_util"] = df["revol_util"].fillna(df["revol_util"].median()).clip(0, 100)

    # c) outliers - cap income at the 99th percentile (winsorising)
    cap = df["annual_income"].quantile(0.99)
    report["income_cap_99th_pct"] = float(cap)
    report["income_values_capped"] = int((df["annual_income"] > cap).sum())
    df["annual_income"] = df["annual_income"].clip(upper=cap)

    # d) target + money columns
    df["is_default"] = (df["loan_status"] == "Charged Off").astype(int)
    df["profit"] = df["total_payment"] - df["loan_amount"]

    report["rows_clean"] = len(df)
    return add_features(df), report


# ----------------------------------------------------------------------------------
# 2. FEATURE ENGINEERING
# ----------------------------------------------------------------------------------
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["grade_idx"] = df["grade"].map({g: i for i, g in enumerate(GRADES)})
    df["loan_to_income"] = df["loan_amount"] / df["annual_income"]
    df["issue_year"] = df["issue_date"].dt.year
    df["issue_quarter"] = df["issue_date"].dt.quarter

    df["dti_band"] = pd.cut(df["dti"], [-0.01, 10, 20, 30, 100],
                            labels=["<10", "10-20", "20-30", "30+"]).astype(str)
    df["util_band"] = pd.cut(df["revol_util"], [-0.01, 30, 60, 80, 100],
                             labels=["<30%", "30-60%", "60-80%", "80%+"]).astype(str)
    df["income_band"] = pd.cut(df["annual_income"], [0, 40_000, 70_000, 100_000, np.inf],
                               labels=["<40k", "40-70k", "70-100k", "100k+"]).astype(str)
    df["emp_band"] = pd.cut(df["emp_length_yrs"], [-1, 1, 5, 9, 10],
                            labels=["<2 yrs", "2-5 yrs", "6-9 yrs", "10+ yrs"]).astype(str)
    return df


# ----------------------------------------------------------------------------------
# 3. RISK MODEL (logistic regression - chosen because it is explainable)
# ----------------------------------------------------------------------------------
def build_model() -> Pipeline:
    pre = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
         CATEGORICAL_FEATURES),
    ])
    return Pipeline([("prep", pre), ("clf", LogisticRegression(max_iter=1000))])


def fit_and_score(df: pd.DataFrame):
    """Train on 70%, evaluate on held-out 30%, then score every loan.

    Returns (scored_df, info) where scored_df has pd_score (probability of default),
    risk_decile (1 = safest ... 10 = riskiest) and risk_band.
    """
    features = NUMERIC_FEATURES + CATEGORICAL_FEATURES
    X, y = df[features], df["is_default"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y)

    model = build_model().fit(X_train, y_train)
    test_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, test_prob)

    # Baseline to beat: just using the lender's grade as the score
    grade_auc = roc_auc_score(y_test, X_test["grade_idx"])

    names = model.named_steps["prep"].get_feature_names_out()
    coefs = model.named_steps["clf"].coef_[0]
    drivers = (pd.DataFrame({"feature": [n.split("__", 1)[1] for n in names],
                             "coefficient": coefs, "odds_ratio": np.exp(coefs)})
               .sort_values("coefficient", ascending=False).reset_index(drop=True))

    scored = df.copy()
    scored["split"] = "train"
    scored.loc[X_test.index, "split"] = "test"
    scored["pd_score"] = model.predict_proba(X)[:, 1]
    rank = scored["pd_score"].rank(method="first")
    scored["risk_decile"] = pd.qcut(rank, 10, labels=range(1, 11)).astype(int)
    scored["risk_band"] = pd.qcut(rank, 5, labels=RISK_BAND_LABELS).astype(str)

    # Expected profit = P(repay) x interest earned  -  P(default) x principal lost
    interest = scored["loan_amount"] * scored["int_rate"] / 100 * scored["term_months"] / 12 * INTEREST_FACTOR
    scored["expected_profit"] = ((1 - scored["pd_score"]) * interest
                                 - scored["pd_score"] * scored["loan_amount"] * LOSS_GIVEN_DEFAULT)

    info = {"auc_model": float(auc), "auc_grade_only": float(grade_auc),
            "n_train": len(X_train), "n_test": len(X_test), "drivers": drivers}
    return scored, info


# ----------------------------------------------------------------------------------
# 4. POLICY SIMULATION  (what happens if we reject a group of applicants?)
# ----------------------------------------------------------------------------------
def simulate_rejection(df: pd.DataFrame, reject_mask: pd.Series) -> dict:
    """Compare the book 'as is' with the book if `reject_mask` loans had been declined."""
    kept, rejected = df[~reject_mask], df[reject_mask]
    base_profit, new_profit = df["profit"].sum(), kept["profit"].sum()
    return {
        "loans_rejected": int(len(rejected)),
        "pct_rejected": len(rejected) / len(df),
        "default_rate_before": df["is_default"].mean(),
        "default_rate_after": kept["is_default"].mean() if len(kept) else np.nan,
        "defaults_avoided": int(rejected["is_default"].sum()),
        "good_loans_lost": int((rejected["is_default"] == 0).sum()),
        "profit_before": base_profit,
        "profit_after": new_profit,
        "profit_change": new_profit - base_profit,
    }

In [ ]:
# make the functions above available as `pl.<name>`, exactly like the scripts do
import types
pl = types.SimpleNamespace(**globals())

## 4. Full analysis: charts, model results, Power BI export files

In [ ]:
import json
from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt
import pandas as pd


OUT = Path.cwd() / "outputs"
CHARTS, PBI = OUT / "charts", OUT / "powerbi"
for d in (CHARTS, PBI):
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold"})
BLUE, RED = "#2F5D8C", "#C0392B"


def bar_default_rate(df, col, title, fname, order=None):
    """Bar chart of default rate for each value of `col`, with the portfolio average line."""
    rate = df.groupby(col)["is_default"].mean().mul(100)
    if order:
        rate = rate.reindex(order)
    else:
        rate = rate.sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(rate.index.astype(str), rate.values, color=BLUE)
    ax.axhline(df["is_default"].mean() * 100, color=RED, ls="--", lw=1.2, label="Portfolio average")
    for i, v in enumerate(rate.values):
        ax.text(i, v + 0.4, f"{v:.1f}%", ha="center", fontsize=8)
    ax.set_title(title)
    ax.set_ylabel("Default rate (%)")
    ax.legend(frameon=False)
    plt.xticks(rotation=30 if len(rate) > 5 else 0, ha="right" if len(rate) > 5 else "center")
    plt.tight_layout()
    fig.savefig(CHARTS / fname)
    plt.close(fig)


def main():
    # ---------------------------------------------------------------- 1. CLEAN
    df, report = pl.load_and_clean()
    print("CLEANING REPORT")
    for k, v in report.items():
        print(f"  {k:<24}{v:,.0f}" if isinstance(v, (int, float)) else f"  {k:<24}{v}")

    # ---------------------------------------------------------------- 2. EDA
    print(f"\nPortfolio: {len(df):,} loans | default rate {df['is_default'].mean():.1%} "
          f"| disbursed ${df['loan_amount'].sum() / 1e6:,.1f}M | net profit ${df['profit'].sum() / 1e6:,.1f}M")

    bar_default_rate(df, "grade", "Default rate by grade", "01_default_by_grade.png", pl.GRADES)
    bar_default_rate(df, "dti_band", "Default rate by DTI band", "02_default_by_dti.png",
                     ["<10", "10-20", "20-30", "30+"])
    bar_default_rate(df, "purpose", "Default rate by loan purpose", "03_default_by_purpose.png")
    bar_default_rate(df, "term_months", "Default rate by term (months)", "04_default_by_term.png")
    bar_default_rate(df, "util_band", "Default rate by credit utilisation", "05_default_by_util.png",
                     ["<30%", "30-60%", "60-80%", "80%+"])

    # Profit per loan by grade: the key business insight (default rate alone is misleading)
    prof = df.groupby("grade").agg(default_rate=("is_default", "mean"),
                                   profit_per_loan=("profit", "mean")).reindex(pl.GRADES)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(prof.index, prof["profit_per_loan"], color=BLUE)
    ax.set_title("Average profit per loan by grade")
    ax.set_ylabel("Profit per loan ($)")
    ax2 = ax.twinx()
    ax2.plot(prof.index, prof["default_rate"] * 100, color=RED, marker="o")
    ax2.set_ylabel("Default rate (%)", color=RED)
    ax2.spines["top"].set_visible(False)
    plt.tight_layout()
    fig.savefig(CHARTS / "06_profit_vs_default_by_grade.png")
    plt.close(fig)

    # ---------------------------------------------------------------- 3. MODEL
    scored, info = pl.fit_and_score(df)
    print(f"\nRISK MODEL (logistic regression, {info['n_train']:,} train / {info['n_test']:,} test)")
    print(f"  AUC of model                    : {info['auc_model']:.3f}")
    print(f"  AUC using lender grade alone    : {info['auc_grade_only']:.3f}  <- baseline to beat")
    print("\n  Top risk-INCREASING drivers (odds ratio > 1):")
    print(info["drivers"].head(5).round(3).to_string(index=False))
    print("\n  Top risk-DECREASING drivers (odds ratio < 1):")
    print(info["drivers"].tail(5).round(3).to_string(index=False))

    decile = (scored.groupby("risk_decile")
              .agg(loans=("loan_id", "count"), avg_pd=("pd_score", "mean"),
                   actual_default_rate=("is_default", "mean"),
                   loan_amount=("loan_amount", "sum"), defaults=("is_default", "sum"),
                   profit=("profit", "sum"), avg_profit_per_loan=("profit", "mean"))
              .reset_index())
    print("\nRISK DECILES (1 = safest, 10 = riskiest) - does predicted risk match reality?")
    print(decile.assign(avg_pd=lambda d: d.avg_pd.round(3),
                        actual_default_rate=lambda d: d.actual_default_rate.round(3),
                        avg_profit_per_loan=lambda d: d.avg_profit_per_loan.round(0))
          [["risk_decile", "loans", "avg_pd", "actual_default_rate", "avg_profit_per_loan"]]
          .to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(decile["risk_decile"], decile["actual_default_rate"] * 100, color=BLUE)
    ax.set_xticks(range(1, 11))
    ax.set_title("Actual default rate by model risk decile")
    ax.set_xlabel("Risk decile (1 = safest, 10 = riskiest)")
    ax.set_ylabel("Default rate (%)")
    plt.tight_layout()
    fig.savefig(CHARTS / "07_default_by_risk_decile.png")
    plt.close(fig)

    # ---------------------------------------------------------------- 4. POLICY SIMULATION
    print("\nPOLICY SIMULATION - 'what if we had rejected these applicants?'")
    rule = (scored["dti"] >= 30) & (scored["emp_length_yrs"] < 2)
    rule_res = pl.simulate_rejection(scored, rule)
    print("  Rule A: reject DTI >= 30 AND employment < 2 yrs")
    for k, v in rule_res.items():
        print(f"    {k:<22}{v:,.4f}" if isinstance(v, float) and abs(v) < 5 else f"    {k:<22}{v:,.0f}")

    cutoff_rows = []
    for top_n in range(0, 6):                      # reject riskiest N deciles
        res = pl.simulate_rejection(scored, scored["risk_decile"] > 10 - top_n)
        cutoff_rows.append({"riskiest_deciles_rejected": top_n, **res})
    cutoffs = pd.DataFrame(cutoff_rows)
    print("\n  Rule B: reject the riskiest N deciles by model score")
    print(cutoffs[["riskiest_deciles_rejected", "pct_rejected", "default_rate_after",
                   "defaults_avoided", "good_loans_lost", "profit_change"]]
          .round(3).to_string(index=False))

    # Rule C: reject only loans the model expects to LOSE money on (expected profit < 0)
    neg = scored["expected_profit"] < 0
    rule_c = pl.simulate_rejection(scored, neg)
    print("\n  Rule C: reject loans with NEGATIVE expected profit (pd x loss > (1-pd) x interest)")
    for k, v in rule_c.items():
        print(f"    {k:<22}{v:,.4f}" if isinstance(v, float) and abs(v) < 5 else f"    {k:<22}{v:,.0f}")

    # Honest check: the model never saw the TEST loans, so this is the number to trust
    test = scored[scored["split"] == "test"]
    rule_c_test = pl.simulate_rejection(test, test["expected_profit"] < 0)
    print("\n  Rule C on HELD-OUT test loans only (the trustworthy estimate):")
    print(f"    loans rejected {rule_c_test['pct_rejected']:.1%} | default rate "
          f"{rule_c_test['default_rate_before']:.2%} -> {rule_c_test['default_rate_after']:.2%} | "
          f"profit change ${rule_c_test['profit_change']:,.0f} "
          f"({rule_c_test['profit_change'] / rule_c_test['profit_before']:+.1%})")

    # ---------------------------------------------------------------- 5. EXPORTS
    scored.drop(columns=["total_payment"]).to_csv(PBI / "loans_scored.csv", index=False)
    decile.to_csv(PBI / "risk_deciles.csv", index=False)
    scored.groupby("grade").agg(loans=("loan_id", "count"), loan_amount=("loan_amount", "sum"),
                                defaults=("is_default", "sum"), profit=("profit", "sum")
                                ).reindex(pl.GRADES).reset_index().to_csv(PBI / "grade_summary.csv", index=False)
    info["drivers"].to_csv(PBI / "model_drivers.csv", index=False)
    cutoffs.to_csv(PBI / "policy_cutoffs.csv", index=False)
    (OUT / "metrics.json").write_text(json.dumps({
        "cleaning": report,
        "portfolio": {"loans": len(df), "default_rate": df["is_default"].mean(),
                      "disbursed": int(df["loan_amount"].sum()), "net_profit": float(df["profit"].sum())},
        "model": {k: v for k, v in info.items() if k != "drivers"},
        "rule_a": {k: float(v) for k, v in rule_res.items()},
        "rule_c": {k: float(v) for k, v in rule_c.items()},
        "rule_c_test_only": {k: float(v) for k, v in rule_c_test.items()},
    }, indent=2, default=float))
    print(f"\nSaved charts -> {CHARTS}\nSaved Power BI files -> {PBI}")

main()

### Charts produced
Default rate by grade / DTI / purpose / term / utilisation, profit vs default by grade, and predicted-vs-actual risk by decile.

In [ ]:
from pathlib import Path
try:
    from IPython.display import Image, display
    for f in sorted(Path('outputs/charts').glob('*.png')):
        print(f.name)
        display(Image(filename=str(f), width=520))
except ImportError:
    print('Charts saved in outputs/charts/')

## 5. Excel what-if workbook
Builds an Excel file made of live formulas. Open it in Excel and change the yellow cells.

In [ ]:
from pathlib import Path

from openpyxl import Workbook
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.worksheet.datavalidation import DataValidation


OUT = Path.cwd() / "outputs" / "Loan_Risk_Policy_Simulator.xlsx"

F = "Arial"
HDR_FILL = PatternFill("solid", fgColor="1F3A5F")
INPUT_FILL = PatternFill("solid", fgColor="FFFF00")
BAND_FILL = PatternFill("solid", fgColor="EAF1F8")
thin = Side(style="thin", color="BBBBBB")
BOX = Border(left=thin, right=thin, top=thin, bottom=thin)
USD, USD2, PCT, INT = '$#,##0;($#,##0);-', '$#,##0.00', '0.0%', '#,##0'


def style_header(ws, row, ncols):
    for c in range(1, ncols + 1):
        cell = ws.cell(row=row, column=c)
        cell.font = Font(name=F, bold=True, color="FFFFFF")
        cell.fill = HDR_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = BOX


def put(ws, ref, value, fmt=None, bold=False, color="000000", fill=None):
    cell = ws[ref]
    cell.value = value
    cell.font = Font(name=F, bold=bold, color=color)
    if fmt:
        cell.number_format = fmt
    if fill:
        cell.fill = fill
    cell.border = BOX
    return cell


def main():
    df, _ = pl.load_and_clean()
    scored, _ = pl.fit_and_score(df)

    grade = (scored.groupby("grade").agg(loans=("loan_id", "count"), amount=("loan_amount", "sum"),
                                         defaults=("is_default", "sum"), profit=("profit", "sum"))
             .reindex(pl.GRADES))
    decile = (scored.groupby("risk_decile").agg(loans=("loan_id", "count"), amount=("loan_amount", "sum"),
                                                defaults=("is_default", "sum"), profit=("profit", "sum")))

    wb = Workbook()

    # ============================================================ Instructions
    ws = wb.active
    ws.title = "Instructions"
    ws.sheet_view.showGridLines = False
    ws["A1"].value = "Loan Risk Policy Simulator"
    ws["A1"].font = Font(name=F, size=16, bold=True)
    lines = [
        "PURPOSE: Test 'what if we only approved loans up to a certain risk level?' and see the effect on default rate and profit.",
        "",
        "HOW TO USE",
        "  1. Go to the 'Policy_Simulator' sheet.",
        "  2. Change the YELLOW cells (max grade approved, or number of risk deciles approved).",
        "  3. Read the results below each input - they update automatically.",
        "",
        "COLOUR LEGEND",
        "  Blue text   = data aggregated from the Python analysis (hard-coded inputs)",
        "  Black text  = Excel formulas (do not overwrite)",
        "  Yellow fill = cells YOU can change",
        "",
        "DATA SOURCE: 50,000 synthetic loans (see project README). Profit = total payments received - loan amount.",
        "Risk deciles come from the logistic-regression model (decile 1 = safest, 10 = riskiest).",
        "Example row: set Max grade = 'E' -> grades F and G are declined; compare default rate and profit vs approving everyone.",
    ]
    for i, t in enumerate(lines, start=3):
        ws.cell(row=i, column=1, value=t).font = Font(name=F, bold=t.isupper() or t.startswith(("HOW", "COLOUR", "PURPOSE", "DATA")))
    ws.column_dimensions["A"].width = 120

    # ============================================================ Grade_Summary
    g = wb.create_sheet("Grade_Summary")
    heads = ["Grade", "Grade Index", "Loans", "Loan Amount ($)", "Defaults", "Profit ($)",
             "Default Rate", "Profit per Loan ($)", "Share of Loans"]
    g["A1"].value = "Portfolio by lender grade"
    g["A1"].font = Font(name=F, size=13, bold=True)
    for c, h in enumerate(heads, start=1):
        g.cell(row=4, column=c, value=h)
    style_header(g, 4, len(heads))
    for i, (grd, row) in enumerate(grade.iterrows()):
        r = 5 + i
        put(g, f"A{r}", grd, bold=True)
        put(g, f"B{r}", i, INT, color="0000FF")
        put(g, f"C{r}", int(row.loans), INT, color="0000FF")
        put(g, f"D{r}", float(row.amount), USD, color="0000FF")
        put(g, f"E{r}", int(row.defaults), INT, color="0000FF")
        put(g, f"F{r}", float(row.profit), USD, color="0000FF")
        put(g, f"G{r}", f"=IFERROR(E{r}/C{r},0)", PCT)
        put(g, f"H{r}", f"=IFERROR(F{r}/C{r},0)", USD)
        put(g, f"I{r}", f"=C{r}/$C$12", PCT)
    put(g, "A12", "Total", bold=True, fill=BAND_FILL)
    put(g, "B12", None, fill=BAND_FILL)
    for col, fmt in zip("CDEF", [INT, USD, INT, USD]):
        put(g, f"{col}12", f"=SUM({col}5:{col}11)", fmt, bold=True, fill=BAND_FILL)
    put(g, "G12", "=IFERROR(E12/C12,0)", PCT, bold=True, fill=BAND_FILL)
    put(g, "H12", "=IFERROR(F12/C12,0)", USD, bold=True, fill=BAND_FILL)
    put(g, "I12", "=SUM(I5:I11)", PCT, bold=True, fill=BAND_FILL)
    for col, w in zip("ABCDEFGHI", [10, 12, 12, 18, 12, 18, 14, 18, 14]):
        g.column_dimensions[col].width = w
    g.row_dimensions[4].height = 32

    ch = BarChart()
    ch.title = "Default rate by grade"
    ch.y_axis.title = "Default rate"
    ch.y_axis.numFmt = "0%"
    ch.add_data(Reference(g, min_col=7, min_row=4, max_row=11), titles_from_data=True)
    ch.set_categories(Reference(g, min_col=1, min_row=5, max_row=11))
    ch.legend = None
    ch.height, ch.width = 7.5, 14
    g.add_chart(ch, "A15")

    # ============================================================ Risk_Deciles
    d = wb.create_sheet("Risk_Deciles")
    heads = ["Risk Decile (1=safest)", "Loans", "Loan Amount ($)", "Defaults", "Profit ($)",
             "Default Rate", "Profit per Loan ($)"]
    d["A1"].value = "Portfolio by model risk decile"
    d["A1"].font = Font(name=F, size=13, bold=True)
    for c, h in enumerate(heads, start=1):
        d.cell(row=4, column=c, value=h)
    style_header(d, 4, len(heads))
    for i, (dec, row) in enumerate(decile.iterrows()):
        r = 5 + i
        put(d, f"A{r}", int(dec), INT, color="0000FF", bold=True)
        put(d, f"B{r}", int(row.loans), INT, color="0000FF")
        put(d, f"C{r}", float(row.amount), USD, color="0000FF")
        put(d, f"D{r}", int(row.defaults), INT, color="0000FF")
        put(d, f"E{r}", float(row.profit), USD, color="0000FF")
        put(d, f"F{r}", f"=IFERROR(D{r}/B{r},0)", PCT)
        put(d, f"G{r}", f"=IFERROR(E{r}/B{r},0)", USD)
    put(d, "A15", "Total", bold=True, fill=BAND_FILL)
    for col, fmt in zip("BCDE", [INT, USD, INT, USD]):
        put(d, f"{col}15", f"=SUM({col}5:{col}14)", fmt, bold=True, fill=BAND_FILL)
    put(d, "F15", "=IFERROR(D15/B15,0)", PCT, bold=True, fill=BAND_FILL)
    put(d, "G15", "=IFERROR(E15/B15,0)", USD, bold=True, fill=BAND_FILL)
    for col, w in zip("ABCDEFG", [22, 12, 18, 12, 18, 14, 18]):
        d.column_dimensions[col].width = w
    d.row_dimensions[4].height = 32

    lc = LineChart()
    lc.title = "Profit per loan by risk decile"
    lc.add_data(Reference(d, min_col=7, min_row=4, max_row=14), titles_from_data=True)
    lc.set_categories(Reference(d, min_col=1, min_row=5, max_row=14))
    lc.legend = None
    lc.height, lc.width = 7.5, 14
    d.add_chart(lc, "A18")

    # ============================================================ Policy_Simulator
    s = wb.create_sheet("Policy_Simulator", 1)
    s.sheet_view.showGridLines = False
    s["A1"].value = "Policy Simulator - what if we tightened approvals?"
    s["A1"].font = Font(name=F, size=15, bold=True)
    s["A2"].value = "Change the yellow cells. Results recalculate automatically."
    s["A2"].font = Font(name=F, italic=True)

    # ---- Scenario 1: by lender grade
    s["A4"].value = "SCENARIO 1: Approve only loans up to a maximum lender grade"
    s["A4"].font = Font(name=F, bold=True, size=12)
    put(s, "A5", "Max grade approved (A-G)", bold=True)
    put(s, "B5", "G", color="0000FF", bold=True, fill=INPUT_FILL)
    dv = DataValidation(type="list", formula1='"A,B,C,D,E,F,G"', allow_blank=False)
    s.add_data_validation(dv)
    dv.add("B5")
    put(s, "A6", "Grade index (helper)")
    put(s, "B6", "=MATCH(B5,Grade_Summary!$A$5:$A$11,0)-1", INT)

    rows1 = [
        ("Loans approved", '=SUMIFS(Grade_Summary!$C$5:$C$11,Grade_Summary!$B$5:$B$11,"<="&B6)', INT),
        ("% of applicants approved", "=B8/Grade_Summary!$C$12", PCT),
        ("Defaults among approved", '=SUMIFS(Grade_Summary!$E$5:$E$11,Grade_Summary!$B$5:$B$11,"<="&B6)', INT),
        ("Default rate (approved book)", "=IFERROR(B10/B8,0)", PCT),
        ("Default rate (approve everyone)", "=Grade_Summary!$G$12", PCT),
        ("Total profit (approved book)", '=SUMIFS(Grade_Summary!$F$5:$F$11,Grade_Summary!$B$5:$B$11,"<="&B6)', USD),
        ("Total profit (approve everyone)", "=Grade_Summary!$F$12", USD),
        ("PROFIT CHANGE vs approve-everyone", "=B13-B14", USD),
    ]
    r = 8
    for label, formula, fmt in rows1:
        put(s, f"A{r}", label, bold=label.startswith("PROFIT"))
        put(s, f"B{r}", formula, fmt, bold=label.startswith("PROFIT"))
        r += 1

    # ---- Scenario 2: by model risk decile
    s["A18"].value = "SCENARIO 2: Approve only the safest N risk deciles (model score)"
    s["A18"].font = Font(name=F, bold=True, size=12)
    put(s, "A19", "Safest deciles approved (1-10)", bold=True)
    put(s, "B19", 9, color="0000FF", bold=True, fill=INPUT_FILL)
    dv2 = DataValidation(type="whole", operator="between", formula1="1", formula2="10")
    s.add_data_validation(dv2)
    dv2.add("B19")
    rows2 = [
        ("Loans approved", '=SUMIFS(Risk_Deciles!$B$5:$B$14,Risk_Deciles!$A$5:$A$14,"<="&B19)', INT),
        ("% of applicants approved", "=B20/Risk_Deciles!$B$15", PCT),
        ("Defaults among approved", '=SUMIFS(Risk_Deciles!$D$5:$D$14,Risk_Deciles!$A$5:$A$14,"<="&B19)', INT),
        ("Default rate (approved book)", "=IFERROR(B22/B20,0)", PCT),
        ("Default rate (approve everyone)", "=Risk_Deciles!$F$15", PCT),
        ("Total profit (approved book)", '=SUMIFS(Risk_Deciles!$E$5:$E$14,Risk_Deciles!$A$5:$A$14,"<="&B19)', USD),
        ("Total profit (approve everyone)", "=Risk_Deciles!$E$15", USD),
        ("PROFIT CHANGE vs approve-everyone", "=B25-B26", USD),
    ]
    r = 20
    for label, formula, fmt in rows2:
        put(s, f"A{r}", label, bold=label.startswith("PROFIT"))
        put(s, f"B{r}", formula, fmt, bold=label.startswith("PROFIT"))
        r += 1

    s["A30"].value = ("How to read it: try deciles = 9 (decline the riskiest 10%) vs 8 or 7. "
                      "Profit usually rises first, then falls once you start rejecting good, interest-paying loans.")
    s["A30"].font = Font(name=F, italic=True)
    s["A30"].alignment = Alignment(wrap_text=True, vertical="top")
    s.merge_cells("A30:D32")
    s.column_dimensions["A"].width = 38
    s.column_dimensions["B"].width = 20

    wb.save(OUT)
    print(f"Saved {OUT}")

main()

## 6. Key findings (50,000 loans, $839M disbursed)

- Default rate rises from **6% (grade A) to 36% (grade G)** and from **8% to 30%** as DTI goes from under 10 to over 30.
- Profit per loan **peaks at grade D (~$2,170)** and falls to ~$520 at grade G: default rate alone is misleading.
- Logistic-regression risk model: **AUC 0.72 vs 0.67** for lender grade alone (held-out test set).
- A simple rule (DTI >= 30 and employed < 2 yrs) affects 0.5% of loans and barely changes profit.
- Declining the riskiest **20%** of applicants *reduces* profit by about $6.4M.
- Declining only loans with **negative expected profit**: declines 6.6% of applicants, default rate **15.4% to 13.6%**, profit **+4.4%** on held-out loans.

**Recommendation:** decline or reprice only where expected loss exceeds expected interest income. Do not tighten across the board.

**Next:** build the Power BI dashboard from `outputs/powerbi/*.csv` using the DAX in `POWERBI_GUIDE.md`; deploy `app.py` on Streamlit Community Cloud.